# Caret ensemble


In [2]:
library(caret)

library(data.table)
library(caretEnsemble)

library(mlbench)
library(pROC)
library(rpart)

library(randomForest)
library(nnet)

library(pbapply)


## Funcións útiles


In [3]:
# Elimina os caracteres non validos para unha
# variable de R dos niveis dun factor
renomear_niveis <- function(fact) {
  # Reemprezamos por un punto calquera caracter
  # non valido para un nome de variable
  levels(fact) <- make.names(levels(fact))

  return(fact)
}

mover_columna <- function(df, nome_columna = NULL) {
  # Obtemos a posicion da variable
  if (!is.null(nome_columna)) {
    i <- which(colnames(df) == nome_columna)
    # Reorganizamos as columnas do data frame se a variable
    # non esta na posicion que queremos
    if (i != ncol(df)) {
      df <- df[, c(seq_len(i - 1), (i + 1):ncol(df), i)]
    }
  }

  return(df)
}


# Renomea a variable dun data frame
renomear_variable <- function(df, nome_novo = "y", nome_columna) {
  # Comprobamos se algunha columna se chama como o
  # nome da variable que queremos renomear
  if (nome_novo %in% colnames(df)) {
    # Buscamos a posicion da columna que queremos renomear
    if (is.null(nome_columna)) {
      pos_col <- ncol(df)
    } else {
      pos_col <- which(names(df) == nome_columna)
    }
    # Renomeamos todas as columnas do df excepto a que queremos renomear
    colnames(df)[-pos_col] <- paste0("V", colnames(df)[-pos_col])
  }

  # Renomeamos a columna que queremos renomear
  if (is.null(nome_columna)) {
    colnames(df)[ncol(df)] <- nome_novo
  } else {
    colnames(df)[colnames(df) == nome_columna] <- nome_novo
  }

  return(df)
}

reducir_niveis <- function(df, clases_positivas, clase_positiva, clase_negativa) {
  # Copiar o df orixinal
  df_novo <- df

  # Asignar o novo nivel aos niveis que non se queren manter
  df_novo$y <- ifelse(df_novo$y %in% clases_positivas,
    clase_positiva, clase_negativa
  )

  # Convertimos a columna nun factor de dous niveis
  df_novo$y <- factor(df_novo$y,
    levels = c(clase_positiva, clase_negativa)
  )

  return(df_novo)
}

# Funcion para ler un arquivo CSV
ler_csv <- function(
    arquivo, sep = ",", header = FALSE,
    variable_resposta = NULL,
    clases_positivas = NULL, clase_positiva = NULL, clase_negativa = NULL) {
  # Le o arquivo CSV
  df <- read.csv(arquivo, sep = sep, header = header, na.strings = "?", stringsAsFactors = TRUE)

  # Eliminamos as filas incompletas (con ?)
  df <- na.omit(df)

  # Movemos a columna da variable resposta ao final
  df <- mover_columna(df, nome_columna = variable_resposta)

  # Renomea a variable correspondete a variable resposta a "y"
  df <- renomear_variable(df, nome_columna = variable_resposta)

  # Modificamos o numero de clases no caso de que se especifique
  if (!is.null(clases_positivas) && !is.null(clase_positiva) && !is.null(clase_negativa)) {
    df <- reducir_niveis(df,
      clases_positivas = clases_positivas,
      clase_positiva = clase_positiva,
      clase_negativa = clase_negativa
    )
  } else {
    df$y <- factor(df$y)
  }
  # Renomea os niveis da variable clase
  df$y <- renomear_niveis(df$y)

  # Eliminamos todas as columnas que sexan de tipo factor excepto a ultima columna
  columnas_factor <- c(names(df[, -ncol(df)])[sapply(df[, -ncol(df)], is.factor)])
  df <- df[, !(names(df) %in% columnas_factor)]

  return(df)
}

# Dividimos o conxunto de partida en adestramento e test
# E necesario que a variable resposta se chame "y"
division_conxunto <- function(df, p = 0.7, seed = NULL) {
  if (!is.null(seed)) {
    set.seed(seed)
  }
  train_index <- caret::createDataPartition(df$y, p = p, list = FALSE)
  train_data <- df[train_index, ]
  test_data <- df[-train_index, ]

  division <- list(
    train_index = train_index,
    train_data = train_data,
    test_data = test_data
  )

  return(division)
}

avaliar <- function(predicions, y) {
  cm <- caret::confusionMatrix(predicions, y)

  nclases <- length(levels(y))

  p <- numeric(nclases)
  r <- numeric(nclases)
  f1 <- numeric(nclases)
  ac <- cm$overall[1]

  for (i in seq_len(nclases)) {
    # dividimos entre a suma da fila para a precision
    p[i] <- cm$table[i, i] / sum(cm$table[i, ])
    # dividimos entre a suma da columna para o recall
    r[i] <- cm$table[i, i] / sum(cm$table[, i])
    f1[i] <- (2 * p[i] * r[i] / (p[i] + r[i]))
  }

  res <- list(
    accuracy = ac,
    metrics = data.frame(Precision = p, Recall = r, FScore = f1), # nolint: line_length_linter.
    metrics_ref_class = data.frame(Precision = p[1], Recall = r[1], FScore = f1[1]) # nolint: line_length_linter.
  )

  return(res)
}

comparar_stacking <- function(model_stack, model_stack_augmented, newdata) {
  pred <- predict(model_stack, newdata = newdata, type = "raw")
  pred_augmented <- predict(model_stack_augmented, newdata = newdata, type = "raw")
  av <- avaliar(pred, newdata$y)
  av_aug <- avaliar(pred_augmented, newdata$y)

  return(list(
    diff_accuracy = av_aug$accuracy - av$accuracy,
    diff_metrics = av_aug$metrics - av$metrics,
    diff_metrics_ref_class = av_aug$metrics_ref_class - av$metrics_ref_class
  ))
}


## Replicamos o exemplo de [CRAN](https://cran.r-project.org/web/packages/caretEnsemble/vignettes/caretEnsemble-intro.html)


Lemos o conxunto de datos e dividimos en train e test


In [4]:
library("mlbench")
data(Sonar)

set.seed(107)

inTrain <- caret::createDataPartition(y = Sonar$Class, p = .75, list = FALSE)
training <- Sonar[inTrain, ]
testing <- Sonar[-inTrain, ]


Definimos un trainControl para o adestramento dos modelos.

- Utilizamos un método de validación cruzada para a selección de hiperparámetros.
- Establecemos a `true` a opción `classProbs` para que nos devolva probabilidades en lugar de predicións.
- Estabelcemos `savePredictions = "final"` para que se garden as predicións finais, en lugar de todas.
- Desordeamos o conxunto de adestramento.


In [6]:
my_control <- trainControl(
  method = "boot",
  number = 25,
  classProbs = TRUE,
  savePredictions = "final",
  index = createResample(training$Class, 25),
  summaryFunction = twoClassSummary
)


Definimos unha lista de modelos con `caretList`. Basicamente, esta función adestra todos os modelos definidos cos hiperparámetros seleccionados e co mesmo método de validación cruzada especificado. É unha forma de unificar o adestramento de varios modelos.


In [8]:
model_list <- caretEnsemble::caretList(
  Class ~ .,
  data = training,
  trControl = my_control,
  methodList = c("glm", "rpart")
)


Warning message in train.default(x, y, weights = w, ...):
"The metric "Accuracy" was not in the result set. ROC will be used instead."
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"

Predicimos con todos os modelos e gardamos o resultado nun dataframe. NOTA: esta función automaticamente selecciona unha das columnas cando calculamos probabilidades. Deste modo, non é necesario especificar a columna de probabilidade (mirei o código fonte e tampouco temos opción de facelo).

Quizais, sería unha boa idea comprobar por que non permite incluír máis dunha clase e, no caso de que teña sentido, permitir ao usuario especificar cantas columnas quere que aparezan na saída (ou cales).

Actualmente, a biblioteca selecciona unha columna co método `getBinaryTargetLevel` unha columna. Polo que vin, este método selecciona a columna correspondente coa clase de referencia nun contexto de clasificación binaria. Polo xeral, se non se especifica unha clase de referencia selecciona a primeira. Probei, e no caso de selección multiclase e tamén selecciona a primeira columna (polo que estaríamos perdendo información, porque só devolve unha clase).


In [12]:
p <- as.data.frame(predict(model_list, newdata = head(testing)))
p


glm,rpart
<dbl>,<dbl>
2.220446e-16,0.8750000
1.000000e+00,0.8750000
1.000000e+00,0.1818182
2.354789e-05,0.1818182
2.220446e-16,0.1818182
1.000000e+00,0.1818182


In [11]:
library("randomForest")
library("nnet")

model_list_big <- caretList(
  Class ~ .,
  data = training,
  trControl = my_control,
  metric = "ROC",
  methodList = c("glm", "rpart"),
  tuneList = list(
    rf1 = caretModelSpec(method = "rf", tuneGrid = data.frame(.mtry = 2)),
    rf2 = caretModelSpec(method = "rf", tuneGrid = data.frame(.mtry = 10), preProcess = "pca"),
    nn = caretModelSpec(method = "nnet", tuneLength = 2, trace = FALSE)
  )
)


randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.


Attaching package: 'randomForest'


The following object is masked from 'package:ggplot2':

    margin


Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit: algorithm did not converge"
Warning message:
"glm.fit: fitted probabilities numerically 0 or 1 occurred"
Warning message:
"glm.fit:

In [13]:
p <- as.data.frame(predict(model_list_big, newdata = testing))
head(p)


,rf1,rf2,nn,glm,rpart
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.434,0.378,0.1762613,2.220446e-16,0.8750000
2,0.676,0.492,0.9692479,1.000000e+00,0.8750000
3,0.554,0.492,0.9161972,1.000000e+00,0.1818182
4,0.482,0.486,0.5023163,2.354789e-05,0.1818182
5,0.218,0.180,0.1500791,2.220446e-16,0.1818182
6,0.460,0.526,0.8877006,1.000000e+00,0.1818182


## Proba co dataset Iris


Probamos a ver que clase devolve nun contexto de clasificación binaria.


In [9]:
data_iris <- ler_csv("./data/iris.data", header = FALSE)
head(data_iris)

division_iris <- division_conxunto(data_iris, p = 0.7, seed = 1234)
table(division_iris$train_data$y)
table(division_iris$test_data$y)


,V1,V2,V3,V4,y
,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,5.1,3.5,1.4,0.2,Iris.setosa
2,4.9,3.0,1.4,0.2,Iris.setosa
3,4.7,3.2,1.3,0.2,Iris.setosa
4,4.6,3.1,1.5,0.2,Iris.setosa
5,5.0,3.6,1.4,0.2,Iris.setosa
6,5.4,3.9,1.7,0.4,Iris.setosa



    Iris.setosa Iris.versicolor  Iris.virginica 
             35              35              35 


    Iris.setosa Iris.versicolor  Iris.virginica 
             15              15              15 

In [10]:
index <- createFolds(division_iris$train_data$y, k = 5, list = TRUE, returnTrain = TRUE)

my_control_probs <- trainControl(
  method = "cv",
  number = 5,
  classProbs = TRUE,
  index = index,
  savePredictions = "final",
)

my_control <- trainControl(
  method = "cv",
  number = 5,
  classProbs = FALSE,
  index = index,
  savePredictions = "final",
)


In [92]:
model_list_probs <- caretList(
  y ~ .,
  data = division_iris$train_data,
  trControl = my_control_probs,
  tuneList = list(
    rf1 = caretModelSpec(method = "rf", tuneGrid = data.frame(.mtry = 2)),
    nn = caretModelSpec(method = "nnet", tuneLength = 2, trace = FALSE)
  )
)

model_list <- caretList(
  y ~ .,
  data = division_iris$train_data,
  trControl = my_control,
  tuneList = list(
    rf1 = caretModelSpec(method = "rf", tuneGrid = data.frame(.mtry = 2)),
    nn = caretModelSpec(method = "nnet", tuneLength = 2, trace = FALSE)
  )
)


In [93]:
p <- as.data.frame(predict(model_list_probs, newdata = division_iris$test_data))
head(p)


,rf1,nn
,<dbl>,<dbl>
1,1,0.9865156
2,1,0.9837858
3,1,0.9810857
4,1,0.9872206
5,1,0.9816708
6,1,0.9888335


In [94]:
p <- as.data.frame(predict(model_list, newdata = division_iris$test_data))
head(p)


,rf1,nn
,<chr>,<chr>
1,Iris.setosa,Iris.setosa
2,Iris.setosa,Iris.setosa
3,Iris.setosa,Iris.setosa
4,Iris.setosa,Iris.setosa
5,Iris.setosa,Iris.setosa
6,Iris.setosa,Iris.setosa


Efectivamente a primeira clase é a clase de referencia, cando non especificamos o contrario


In [76]:
levels(division_iris$train_data$y)[1]


[1] "Iris.setosa"

## Probamos cun dataset con dúas clases


In [11]:
data_dd1 <- ler_csv("./data/dd1_1.csv", header = TRUE, variable_resposta = "class")
head(data_dd1)

division_dd1 <- division_conxunto(data_dd1, p = 0.7, seed = 1234)
table(division_dd1$train_data$y)
table(division_dd1$test_data$y)


,x1,x2,y
,<dbl>,<dbl>,<fct>
1,-1.2484289,2.1606268,X1
2,-0.8685649,-0.4759967,X1
3,0.1966088,1.5610417,X1
4,2.3185670,-1.0302444,X1
5,0.3326163,0.8700212,X1
6,-0.1734771,-0.2000962,X1



 X1  X2 
210 210 


X1 X2 
90 90 

In [12]:
trControl <- trainControl(
  method = "cv",
  number = 5,
  classProbs = TRUE,
  index = createFolds(division_dd1$train_data$y, k = 5, list = TRUE, returnTrain = TRUE),
  savePredictions = "final",
)


In [13]:
model_list <- caretList(
  y ~ .,
  data = division_dd1$train_data,
  trControl = trControl,
  tuneList = list(
    rf = caretModelSpec(method = "rf", tuneGrid = data.frame(.mtry = 2)),
    nn = caretModelSpec(method = "nnet", tuneLength = 2, trace = FALSE)
  )
)


Nesta biblioteca adestran o modelo con todo o conxunto de train. Deste modo, ao calcular as predicións sobre este mesmo conxunto poderíamos estar facendo overfitting. Supoño que canto máis grande sexa o conxunto de train, este problema desaparecerá. Ademais, observando as probabilidades de pertenza á primeira clase cos métodos `rf` e `nnet` vemos que non son 1 nin está excesivamente preto. Imaxino que isto pode ser un indicativo de que non está habendo problema con isto.


In [14]:
probs <- list(
  train = as.data.frame(predict(model_list, newdata = division_dd1$train_data)),
  test = as.data.frame(predict(model_list, newdata = division_dd1$test_data))
)

head(probs$train)
head(probs$test)


,rf,nn
,<dbl>,<dbl>
1,0.804,0.7355138
2,0.804,0.5391173
3,0.984,0.7356858
4,0.980,0.6905726
5,0.754,0.6262949
6,0.738,0.5415384


,rf,nn
,<dbl>,<dbl>
1,0.242,0.6703746
2,0.990,0.7198988
3,0.406,0.7430667
4,0.650,0.7157855
5,0.532,0.5642386
6,0.678,0.5418321


Ampliamos ambos conxuntos de datos coas probabilidades que acabamos de obter.


In [101]:
data_am <- list(train_data = cbind(division_dd1$train_data, probs$train), test_data = cbind(division_dd1$test_data, probs$test))

head(data_am$train_data)
head(data_am$test_data)


,x1,x2,y,rf,nn
,<dbl>,<dbl>,<fct>,<dbl>,<dbl>
3,0.1966088,1.5610417,X1,0.798,0.7310257
4,2.3185670,-1.0302444,X1,0.784,0.5398729
6,-0.1734771,-0.2000962,X1,0.992,0.7332381
7,-1.7672545,1.3372535,X1,0.960,0.6905185
8,2.1044394,0.1750016,X1,0.730,0.6234921
9,3.2498704,1.8301059,X1,0.758,0.5387038


,x1,x2,y,rf,nn
,<dbl>,<dbl>,<fct>,<dbl>,<dbl>
1,-1.2484289,2.1606268,X1,0.222,0.6673147
2,-0.8685649,-0.4759967,X1,0.990,0.7185454
5,0.3326163,0.8700212,X1,0.414,0.7389889
13,0.7715204,-0.1505199,X1,0.688,0.7127883
14,-0.1738622,3.5774598,X1,0.512,0.5586100
16,2.9571718,0.7080986,X1,0.682,0.5392680


Adestramos un modelo sobre o dataset ampliado e avaliamos


In [111]:
library(e1071) # SVM
library(kernlab) # SVM



Attaching package: 'kernlab'


The following object is masked from 'package:ggplot2':

    alpha




In [135]:
trControl <- trainControl(
  method = "cv", number = 10, classProbs = TRUE
)

tuneGrid <- expand.grid(C = c(0.1, 1, 10, 100, 1000))

svm_model <- train(
  y ~ .,
  data = data_am$train_data,
  method = "svmLinear",
  tuneGrid = tuneGrid,
  trControl = trControl,
)


In [139]:
predicions <- predict(svm_model, data_am$test_data)
predicions


[1] X2 X1 X2 X1 X1 X1 X1 X1 X2 X2 X2 X2 X2 X1 X1 X1 X1 X1 X1 X1 X1 X1 X1 X2 X1
 [26] X1 X1 X1 X1 X2 X1 X1 X1 X2 X2 X2 X1 X2 X2 X1 X2 X2 X1 X2 X1 X1 X1 X1 X1 X2
 [51] X2 X1 X1 X1 X2 X1 X1 X2 X2 X2 X1 X2 X1 X2 X1 X2 X2 X2 X2 X1 X1 X2 X1 X2 X1
 [76] X1 X1 X1 X2 X1 X1 X2 X1 X1 X2 X2 X2 X1 X2 X1 X1 X2 X2 X2 X2 X2 X1 X2 X1 X1
[101] X2 X2 X2 X1 X1 X2 X2 X1 X1 X1 X2 X1 X2 X2 X1 X1 X2 X2 X2 X1 X1 X2 X2 X1 X2
[126] X2 X1 X2 X2 X2 X2 X2 X2 X2 X2 X2 X2 X2 X1 X2 X2 X2 X2 X2 X1 X1 X2 X2 X1 X1
[151] X2 X1 X2 X1 X2 X1 X2 X2 X2 X1 X2 X1 X1 X1 X2 X1 X1 X2 X2 X2 X2 X2 X2 X2 X1
[176] X1 X1 X2 X2 X1
Levels: X1 X2

In [142]:
avaliacion <- avaliar(predicions, data_am$test_data$y)
paste0("Accuracy: ", avaliacion$accuracy)
avaliacion$medidores_clase_positiva


[1] "Accuracy: 0.605555555555556"

Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
0.6091954,0.5888889,0.5988701


### Comprobamos se mellora o rendemento con respecto a empregar `caretStack`


In [143]:
svm_stack <- caretStack(
  model_list,
  method = "svmLinear",
  verbose = FALSE,
  tuneLength = 10,
  trControl = trainControl(
    method = "cv",
    number = 10,
    savePredictions = "final",
    classProbs = TRUE
  )
)


In [148]:
stack_predictions <- predict(svm_stack, newdata = data_am$test_data)


In [149]:
avaliacion_stack <- avaliar(stack_predictions, data_am$test_data$y)
paste0("Accuracy: ", avaliacion_stack$accuracy)
avaliacion_stack$metrics_ref_class


[1] "Accuracy: 0.605555555555556"

Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
0.5940594,0.6666667,0.6282723


## Función caretStackAugmented


### Continuacion


Copiamos a función varias funcións do paquete de `caretEnsemble`. Explicamos o funcionamento delas:

### bestPreds

Básicamente, esta función obtén as probabilidades asociadas á configuración de hiperparámetros que mellor resultado proporcionou durante a fase de validación cruzada para un determinado modelo.

- `a <- data.table::data.table(x$bestTune, key = names(x$bestTune))`: Crea un data.table a partir dun data.table (ou dun data.frame, xa que data.table é unha extensión de data.frame) coa información dos hiperparámetros óptimos para cada modelo. `x$bestTune` proporciona un dataframe coa configuración óptima dos hiperparámetros para cada modelo.
- `b <- data.table::data.table(x$pred, key = names(x$pred))`: Crea un data.table a partir dun data.table (ou dun data.frame, xa que data.table é unha extensión de data.frame) coa información das predicións de cada modelo. `x$pred` proporciona un dataframe coas predicións ou probabilidades obtidas durante a validación cruzada. No caso de que o argumento `savePredictions` sexa `final`, proporciona só as probabilidades do modelo final (`bestTune`).
- `b <- b[a, ]`: No caso de que `savePredictions` sexa `all`, debemos seleccionar só as predicións correspondentes ós hiperparámetros óptimos. Para iso, facemos unha unión entre `a` e `b` para seleccionar só as filas de `b` que teñan os mesmos hiperparámetros que `a`. Para esta operación axudámonos do argumento `key`de `data.table` e podemos filtrar as filas de `b` con `a`.

### extractBestPreds

Esta función é un wrapper de `bestPreds` que se encarga de extraer as predicións óptimas para cada modelo. A súa saída é unha lista de `data.tables`, onde cada elemento corresponde a un modelo.

### makePredObs

Esta función devolve as probabilidades asociadas a cada unha das clases para cada modelo e as observacións reais. Na biblioteca `caretEnsemble`, esta función soamente devolvía as probabilidades asociadas á clase positiva (por defecto a primeira clase) e as observacións. Por exemplo, no caso dun dataset con dúas clases: `X1` e `X2` e dun ensemble con dous modelos: `rf` e `nnet`, a saída sería a seguinte:

| X1_rf | X2_rf | X1_nn    | X2_nn    |
| ----- | ----- | -------- | -------- |
| 0.472 | 0.528 | 0.712842 | 0.287158 |
| 0.434 | 0.566 | 0.497973 | 0.502028 |

mentres que a saída de `makePredObs` da biblioteca `caretEnsemble` sería:

| rf    | nn       |
| ----- | -------- |
| 0.472 | 0.712842 |
| 0.434 | 0.497973 |


In [4]:
bestPreds <- function(x) {
  stopifnot(is(x, "train"))
  stopifnot({
    x$control$savePredictions %in% c("all", "final") |
      x$control$savePredictions == TRUE
  })
  # Seleccionamos os hiperparametros asociados a
  # configuracion que mellor rendemento obtivo
  a <- data.table::data.table(x$bestTune, key = names(x$bestTune))
  b <- data.table::data.table(x$pred, key = names(x$bestTune))
  b <- b[a, ]
  gc(reset = TRUE) # garbage collection
  # ordenamos b (non crea unha copia) por resample e logo por rowIndex
  data.table::setorderv(b, c("Resample", "rowIndex"))
}

validateCustomModel <- function(x) {
  if (is.null(x$method)) {
    stop(paste(
      "Custom models must be defined with a \"method\" attribute containing the name",
      "by which that model should be referenced.  Example: my.glm.model$method <- \"custom_glm\""
    ))
  }
  x
}

extractModelName <- function(x) {
  if (is.list(x$method)) {
    validateCustomModel(x$method)$method
  } else if (x$method == "custom") {
    validateCustomModel(x$modelInfo)$method
  } else {
    x$method
  }
}

extractBestPreds <- function(list_of_models) {
  out <- lapply(list_of_models, bestPreds)
  if (is.null(names(out))) {
    names(out) <- make.names(sapply(list_of_models, extractModelName), unique = TRUE)
  }
  gc(reset = TRUE)
  return(out)
}

extractModelTypes <- function(list_of_models) {
  types <- sapply(list_of_models, function(x) x$modelType)
  type <- types[1]

  stopifnot(all(types == type))
  stopifnot(all(types %in% c("Classification", "Regression")))
  return(type)
}

makePredObsMatrix2 <- function(list_of_models) {
  modelLibrary <- extractBestPreds(list_of_models)
  # Extract model type (class or reg)
  type <- extractModelTypes(list_of_models)

  # Calculate number of clases of the dataset
  nclases <- length(levels(modelLibrary[[1]]$obs))
  # Quitamos unha clase (non imos incluir as probabilidades asociadas a esa clase)
  classes <- levels(modelLibrary[[1]]$obs)[-nclases]
  # Calculate names of the variables in the final predictions matrix
  class_modelname_combinations <- expand.grid(classes, names(modelLibrary))
  variable_names <- apply(class_modelname_combinations, 1, function(x) paste(x[2], x[1], sep = "_"))

  # Engadimos unha columna extra co nome do modelo
  for (i in seq_along(modelLibrary)) {
    data.table::set(modelLibrary[[i]], j = "modelname", value = names(modelLibrary)[[i]])
  }

  # Quitamos as columnas correspondentes cos hiperparametros
  # Quedamonos coas columnas comuns de todos os data.tables
  keep <- Reduce(intersect, lapply(modelLibrary, names))
  for (i in seq_along(modelLibrary)) { # Percorremos os modelos
    # Quedamonos coas columnas de cada modelo correspondentes cos hiperparametros
    rem <- setdiff(names(modelLibrary[[i]]), keep)
    if (length(rem) > 0) {
      for (r in rem) {
        data.table::set(modelLibrary[[i]], j = r, value = NULL)
      }
    }
  }

  # Xuntamos os data.table asociados a cada modelo nun so data.table
  # Coa columna modelname diferenciamos os modelos
  modelLibrary <- data.table::rbindlist(modelLibrary, fill = TRUE)

  # Reshape wide for meta-modeling
  modelLibrary <- data.table::dcast(
    data = modelLibrary,
    formula = obs + rowIndex + Resample ~ modelname,
    value.var = c(levels(modelLibrary$obs), "pred")
  )


  return(list(obs = modelLibrary$obs, preds = as.matrix(modelLibrary[, variable_names, with = FALSE]), type = type))
}

extractOriginalFeatures <- function(all.models) {
  if (length(all.models) == 0) {
    stop("No models to extract features from")
  }
  trainingData <- all.models[[1]]$trainingData
  return(trainingData[, setdiff(names(trainingData), ".outcome")])
}

check_use_features <- function(use_features, names_original_features) {
  if (!is.null(use_features) && length(use_features) > 0) {
    if (is.character(use_features)) {
      if (any(!use_features %in% names_original_features)) {
        stop("Some features are not present in the original dataset")
      }
    } else if (is.numeric(use_features)) {
      if (any(use_features < 1 | use_features > length(names_original_features))) {
        stop("Some features are not present in the original dataset")
      } else if (any(duplicated(use_features))) {
        stop("Some features are duplicated")
      }
    } else {
      stop("use_features must be a character or numeric vector")
    }
  }
  return(invisible(NULL))
}

caretStackAugmented <- function(all.models, use_features = NULL, ...) {
  predobs <- makePredObsMatrix2(all.models)

  # Build the augmented dataset
  training_data <- predobs$preds
  class_out <- "caretStack"
  if (!is.null(use_features) && length(use_features) > 0) {
    original_features <- extractOriginalFeatures(all.models)
    check_use_features(use_features, names(original_features))
    training_data <- cbind(original_features[, use_features], training_data)
    class_out <- "caretStackAugmented"
  }

  # Build a caret model
  model <- caret::train(training_data, predobs$obs, ...)

  # Return final model
  out <- list(models = all.models, ens_model = model, error = model$results, trainingData = training_data)
  class(out) <- class_out
  return(out)
}

predict2.caretList <- function(object, newdata = NULL, ..., verbose = FALSE) {
  if (is.null(newdata)) {
    warning("Predicting without new data is not well supported.  Attempting to predict on the training data.")
    newdata <- object[[1]]$trainingData
    if (is.null(newdata)) {
      stop("Could not find training data in the first model in the ensemble.")
    }
  }

  if (verbose == TRUE) {
    pbapply::pboptions(type = "txt", char = "*")
  } else if (verbose == FALSE) {
    pbapply::pboptions(type = "none")
  }
  preds <- pbapply::pbsapply(object, function(x) {
    type <- x$modelType
    if (type == "Classification") {
      if (x$control$classProbs) {
        # Return probability predictions for only one of the classes
        # as determined by configured default response class level
        caret::predict.train(x, type = "prob", newdata = newdata, ...)
      } else {
        caret::predict.train(x, type = "raw", newdata = newdata, ...)
      }
    } else if (type == "Regression") {
      caret::predict.train(x, type = "raw", newdata = newdata, ...)
    } else {
      stop(paste("Unknown model type:", type))
    }
  })
  if (!inherits(preds, "matrix") && !inherits(preds, "data.frame")) {
    if (inherits(preds, "character") || inherits(preds, "factor")) {
      preds <- as.character(preds) # drop factorization
    }
    preds <- as.matrix(t(preds))
  }

  if (is.null(names(object))) {
    # If the model list used for predictions is not currently named,
    # then exctract the model names from each model individually.
    # Note that this should only be possible when caretList objects
    # are created manually
    predcols <- sapply(object, extractModelName)
    colnames(preds) <- make.names(predcols, unique = TRUE)
  } else {
    # Otherwise, assign the names of the prediction columns to be
    # equal to the names in the given model list
    colnames(preds) <- names(object)
  }

  return(preds)
}

predict.caretStackAugmented <- function(
    object, newdata = NULL,
    se = FALSE, level = 0.95,
    return_weights = FALSE,
    na.action = na.omit,
    ...) {
  stopifnot(is(object$models, "caretList"))
  type <- extractModelTypes(object$models)

  # Obtemos as predicions de cada modelo con caretList.predict
  probs <- predict2.caretList(object$models, newdata = newdata, na.action = na.action)
  # Construimos un unico datatable
  list_probs_model <- list()
  for (j in seq_len(ncol(probs))) {
    index <- seq_along(unlist(probs[1, j]))
    modelname <- rep(colnames(probs)[j], length(unlist(probs[1, j])))
    probs_model <- data.frame(probs[, j])
    list_probs_model[[colnames(probs)[j]]] <- data.table::data.table(index, probs_model, modelname)
  }
  probs <- data.table::rbindlist(list_probs_model)

  probs <- data.table::dcast(
    data = probs,
    formula = index ~ modelname,
    value.var = c(levels(as.factor(newdata$y)))
  )

  # Paste the original dataset and select and reorder
  # the variables as in the training dataset
  probs <- cbind(newdata, probs)
  probs <- probs[, colnames(object$trainingData)]

  if (type == "Classification") {
    est <- predict(object$ens_model, newdata = probs, na.action = na.action, ...)
  } else {
    est <- predict(object$ens_model, newdata = probs, ...)
  }
  # TODO: weights importance
  out <- est
  return(out)
}


### Probamos o rendemento en comparacion con caretStack e caretEnsemble


In [4]:
data_dd1 <- ler_csv("./data/dd1_1.csv", header = TRUE, variable_resposta = "class")
head(data_dd1)

division_dd1 <- division_conxunto(data_dd1, p = 0.7, seed = 1234)
table(division_dd1$train_data$y)
table(division_dd1$test_data$y)


,x1,x2,y
,<dbl>,<dbl>,<fct>
1,-1.2484289,2.1606268,X1
2,-0.8685649,-0.4759967,X1
3,0.1966088,1.5610417,X1
4,2.3185670,-1.0302444,X1
5,0.3326163,0.8700212,X1
6,-0.1734771,-0.2000962,X1



 X1  X2 
210 210 


X1 X2 
90 90 

In [5]:
trControl <- trainControl(
  method = "cv",
  number = 10,
  classProbs = TRUE,
  index = createFolds(division_dd1$train_data$y, k = 5, list = TRUE, returnTrain = TRUE),
  savePredictions = "final",
)

nnGrid <- expand.grid(
  size = seq(from = 2, to = 10, by = 2),
  decay = seq(from = 0.1, to = 0.5, by = 0.1)
)
rfGrid <- expand.grid(.mtry = ceiling(sqrt(ncol(division_dd1$train_data) - 1)))

model_list <- caretList(
  y ~ .,
  data = division_dd1$train_data,
  trControl = trControl,
  tuneList = list(
    rf = caretModelSpec(method = "rf", tuneGrid = rfGrid),
    nn = caretModelSpec(method = "nnet", tuneGrid = nnGrid, trace = FALSE)
  )
)


In [40]:
names(model_list[[1]])
model_list[[1]]$levels


[1] "method"       "modelInfo"    "modelType"    "results"      "pred"        
 [6] "bestTune"     "call"         "dots"         "metric"       "control"     
[11] "finalModel"   "preProcess"   "trainingData" "ptype"        "resample"    
[16] "resampledCM"  "perfNames"    "maximize"     "yLimits"      "times"       
[21] "levels"       "terms"        "coefnames"    "xlevels"

[1] "X1" "X2"
attr(,"ordered")
[1] FALSE

In [41]:
object <- model_list
newdata <- division_dd1$test_data

probs <- predict2.caretList(model_list, newdata = division_dd1$test_data)


if (is.null(names(object))) {
  # If the model list used for predictions is not currently named,
  # then exctract the model names from each model individually.
  # Note that this should only be possible when caretList objects
  # are created manually
  modelnames <- make.names(sapply(object, extractModelName), unique = TRUE)
} else {
  # Otherwise, assign the names of the prediction columns
  # using the names in the given model list
  modelnames <- names(object)
}

# list of lists of probabilities associated to a model
list_probs_model <- list()
indexes <- seq_len(nrow(newdata)) # rows of the dataset
classes <- object[[1]]$levels

for (j in seq_len(ncol(probs))) {
  modelname <- rep(modelnames[j], length(indexes))
  class_probabilities <- data.frame(probs[, j])
  list_probs_model[[modelnames[j]]] <- data.table::data.table(
    index = indexes, class_probabilities, modelname = modelname
  )
}
probs <- data.table::rbindlist(list_probs_model)

# reshape data from long to wide
probs <- data.table::dcast(
  data = probs,
  formula = index ~ modelname,
  value.var = classes
)
head(probs)

probs <- probs[, 2:ncol(probs)] # remove index column

# Rename columns
rename_cols <- function(nombre) {
  partes <- strsplit(nombre, "_", fixed = TRUE)[[1]]
  nuevo_nombre <- paste0(partes[2], "_", partes[1])
  return(nuevo_nombre)
}
new_colnames <- sapply(colnames(probs), rename_cols)
colnames(probs) <- new_colnames

ordered_colnames <- c()
# Reorder columns
for (modelname in modelnames) {
  modelname_combinations <- expand.grid(modelname, classes)
  ordered_colnames <- c(ordered_colnames, paste(modelname_combinations$Var1, modelname_combinations$Var2, sep = "_"))
}
data.table::setcolorder(probs, ordered_colnames)

head(probs)


index,X1_nn,X1_rf,X2_nn,X2_rf
<int>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.6327948,0.242,0.3672052,0.758
2,0.7212851,0.990,0.2787149,0.010
3,0.7291015,0.406,0.2708985,0.594
4,0.7032173,0.650,0.2967827,0.350
5,0.5464306,0.532,0.4535694,0.468
6,0.5294441,0.678,0.4705559,0.322


rf_X1,rf_X2,nn_X1,nn_X2
<dbl>,<dbl>,<dbl>,<dbl>
0.242,0.758,0.6327948,0.3672052
0.990,0.010,0.7212851,0.2787149
0.406,0.594,0.7291015,0.2708985
0.650,0.350,0.7032173,0.2967827
0.532,0.468,0.5464306,0.4535694
0.678,0.322,0.5294441,0.4705559


In [85]:
tuneGrid <- expand.grid(C = c(0.1, 1, 10))
trControl <- trainControl(method = "cv", number = 5, classProbs = TRUE)

model_stack <- caretEnsemble::caretStack(model_list,
  method = "svmLinear",
  tuneGrid = tuneGrid,
  trControl = trControl
)

model_stack2 <- caretStackAugmented(model_list,
  use_features = NULL, method = "svmLinear",
  tuneGrid = tuneGrid,
  trControl = trControl
)

model_stack_augmented <- caretStackAugmented(model_list,
  use_features = names(division_dd1$train_data[, setdiff(names(division_dd1$train_data), "y")]),
  method = "svmLinear",
  tuneGrid = tuneGrid,
  trControl = trControl
)


In [33]:
probs_stack <- predict(model_stack, newdata = division_dd1$test_data, type = "prob")
preds_stack <- predict(model_stack, newdata = division_dd1$test_data, type = "raw")
probs_stack2 <- predict(model_stack2, newdata = division_dd1$test_data, type = "prob")
preds_stack2 <- predict(model_stack2, newdata = division_dd1$test_data, type = "raw")
probs_stack_aug <- predict(model_stack_augmented, newdata = division_dd1$test_data, type = "prob")
preds_stack_aug <- predict(model_stack_augmented, newdata = division_dd1$test_data, type = "raw")


Os resultados poden variar xa que, cada vez que adestramos un modelo, hai unha compoñente aleatoria pequena que varía os parámetros. Con todo, revisei que o que estivese facendo dese os mesmos resultados intermedios que o código da biblioteca caretEnsemble e podo garantir que, no caso da clasificación binaria e con `use_features = NULL`, estou facendo exactamente o mesmo que a biblioteca `caretEnsemble`.


In [88]:
avaliar(preds_stack_aug, division_dd1$test_data$y)


Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
0.5858586,0.6444444,0.6137566
0.6049383,0.5444444,0.5730994
Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
0.5858586,0.6444444,0.6137566


In [91]:
avaliar(preds_stack2, division_dd1$test_data$y)


Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
0.600,0.6666667,0.6315789
0.625,0.5555556,0.5882353
Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
0.6,0.6666667,0.6315789


In [96]:
comparar_stacking(model_stack, model_stack_augmented, division_dd1$test_data)


Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
-0.01010101,-0.01111111,-0.01058201
-0.01234568,-0.01111111,-0.01169591
Precision,Recall,FScore
<dbl>,<dbl>,<dbl>
-0.01010101,-0.01111111,-0.01058201


In [2]:
library(caret)
library(caretEnsemble)

train <- twoClassSim(
  n = 200, intercept = -8, linearVars = 3,
  noiseVars = 10, corrVars = 4, corrValue = 0.6
)
test <- twoClassSim(
  n = 100, intercept = -7, linearVars = 3,
  noiseVars = 10, corrVars = 4, corrValue = 0.6
)


In [5]:
trControl <- trainControl(
  method = "cv",
  number = 10,
  classProbs = TRUE,
  index = createFolds(train$Class, k = 10, list = TRUE, returnTrain = TRUE),
  savePredictions = "final",
)

nnGrid <- expand.grid(
  size = seq(from = 6, to = 10, by = 2),
  decay = seq(from = 0.1, to = 0.3, by = 0.2)
)
rfGrid <- expand.grid(.mtry = ceiling(sqrt(ncol(train) - 1)))

model_list <- caretList(
  x = train[, -23],
  y = train[, 23],
  trControl = trControl,
  tuneList = list(
    rf = caretModelSpec(method = "rf", tuneGrid = rfGrid),
    nn = caretModelSpec(method = "nnet", tuneGrid = nnGrid, trace = FALSE)
  )
)


In [11]:
trControl <- trainControl(
  method = "cv",
  number = 10,
  classProbs = TRUE,
  index = createFolds(train$Class, k = 10, list = TRUE, returnTrain = TRUE),
  savePredictions = "final",
)

model_stack <- caretStack(model_list, method = "knn", trControl = trControl)

preds <- predict(model_stack, newdata = test, type = "prob", se = TRUE, return_weights = TRUE)
preds


ERROR: Error in names(weights) <- row.names(imp): 'names' attribute [2] must be the same length as the vector [0]
